In [ ]:
import xarray as xr

nc_path = "./S5P_NRTI_L2__NO2____20260319T080229_20260319T080729_43682_03_020901_20260319T084248.nc"

ds = xr.open_dataset(nc_path, group="PRODUCT", decode_cf=False)

print(ds)


<xarray.Dataset> Size: 28MB
Dimensions:                                               (scanline: 373,
                                                           ground_pixel: 450,
                                                           time: 1, corner: 4,
                                                           polynomial_exponents: 6,
                                                           intensity_offset_polynomial_exponents: 1,
                                                           layer: 34,
                                                           vertices: 2)
Coordinates:
  * scanline                                              (scanline) int32 1kB ...
  * ground_pixel                                          (ground_pixel) int32 2kB ...
  * time                                                  (time) int32 4B 511...
  * corner                                                (corner) int32 16B ...
  * polynomial_exponents                                  (polynomial

In [11]:
print(ds.dims)
list(ds.data_vars)
print(ds.data_vars)

FrozenMappingWarningOnValuesAccess({'scanline': 373, 'ground_pixel': 450, 'time': 1, 'corner': 4, 'polynomial_exponents': 6, 'intensity_offset_polynomial_exponents': 1, 'layer': 34, 'vertices': 2})
Data variables:
    latitude                                              (time, scanline, ground_pixel) float32 671kB ...
    longitude                                             (time, scanline, ground_pixel) float32 671kB ...
    delta_time                                            (time, scanline) int32 1kB ...
    time_utc                                              (time, scanline) <U27 40kB ...
    qa_value                                              (time, scanline, ground_pixel) uint8 168kB ...
    nitrogendioxide_tropospheric_column                   (time, scanline, ground_pixel) float32 671kB ...
    nitrogendioxide_tropospheric_column_precision         (time, scanline, ground_pixel) float32 671kB ...
    nitrogendioxide_tropospheric_column_precision_kernel  (time, scanline, 

In [22]:
import pandas as pd

# Extract variables (remove time dimension)
lat = ds["latitude"][0]
lon = ds["longitude"][0]
no2 = ds["nitrogendioxide_tropospheric_column"][0]
qa = ds["qa_value"][0]

# Flatten everything
df = pd.DataFrame({
    "latitude": lat.values.flatten(),
    "longitude": lon.values.flatten(),
    "no2": no2.values.flatten(),
    "qa": qa.values.flatten()
})

df = df[df["qa"] > 0.75]
df = df.dropna()
df.head(50)

,latitude,longitude,no2,qa
1,12.433867,69.219337,1.426741e-05,100
2,12.468092,69.296928,1.016390e-05,100
3,12.501830,69.373505,3.972017e-06,100
4,12.535094,69.449074,8.676368e-06,100
5,12.567896,69.523674,8.726422e-06,100
6,12.600249,69.597328,5.444170e-06,100
7,12.632163,69.670059,7.944590e-06,100
8,12.663649,69.741890,-8.696404e-07,100
9,12.694717,69.812843,1.144983e-05,100
10,12.725379,69.882942,2.292527e-05,100


In [ ]:
lat = 25.09
lon = 76.92
error = 0.01 # if too liberal, you get too many datapoints nearby. If too strict, lat/lon match might not exist in the dataset.
mask0 = (
    (ds.latitude <= lat+error) & (ds.latitude >= lat-error) &
    (ds.longitude <= lon+error) & (ds.longitude >= lon-error)
)
new_no2 = ds['nitrogendioxide_tropospheric_column'].where(mask0, drop=True)

if new_no2.size > 0:
    conc = float(new_no2[0, 0, 0])
    print(f"The concentration of NO2 around ({lat}, {lon}) was {conc} mol/m^2.")
    print("This was found using just lat/lon and error-window matching.")
else:
    # Fallback: find nearest pixel in the swath
    dist2 = ((ds.latitude - lat) ** 2 + (ds.longitude - lon) ** 2).isel(time=0)
    flat_idx = int(dist2.argmin().values)
    s_idx, g_idx = np.unravel_index(flat_idx, dist2.shape)

    nearest_val = ds['nitrogendioxide_tropospheric_column'].isel(time=0, scanline=s_idx, ground_pixel=g_idx)
    nearest_lat = float(ds.latitude.isel(time=0, scanline=s_idx, ground_pixel=g_idx))
    nearest_lon = float(ds.longitude.isel(time=0, scanline=s_idx, ground_pixel=g_idx))
    conc = float(nearest_val)

    print(f"No exact match found within ±{error}°. Using nearest pixel instead.")
    print(f"Nearest coordinates: ({nearest_lat:.4f}, {nearest_lon:.4f})")
    print(f"The concentration of NO2 is {conc} mol/m^2.")
